# MCP LLM Smoke Test
This notebook starts the local MCP server, connects via SSE, exposes the MCP tools to Ollama chat, and executes any returned tool calls. Use it to verify true LLM-driven tool calling before debugging the full pipeline.

In [6]:
import asyncio, subprocess, sys, time, json, os
from mcp.client.sse import sse_client
from mcp import ClientSession

async def start_mcp_server(script_path='mcp/server.py'):
    # Start MCP server as a subprocess; returns the Popen object
    if not os.path.exists(script_path):
        raise FileNotFoundError(f'MCP server script not found: {script_path}')
    proc = subprocess.Popen([sys.executable, script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    # Give server a moment to start
    await asyncio.sleep(2)
    return proc

async def connect_to_mcp(server_url='http://127.0.0.1:8000/sse'):
    ctx = sse_client(server_url)
    read_stream, write_stream = await ctx.__aenter__()
    session = ClientSession(read_stream, write_stream)
    await session.__aenter__()
    try:
        await asyncio.wait_for(session.initialize(), timeout=10)
    except asyncio.TimeoutError:
        await session.__aexit__(None, None, None)
        await ctx.__aexit__(None, None, None)
        raise
    return session, ctx

async def call_tool(session: ClientSession, tool_name: str, arguments: dict) -> dict:
    result = await session.call_tool(tool_name, arguments=arguments)
    if result.content:
        text = result.content[0].text if hasattr(result.content[0], 'text') else str(result.content[0])
        try:
            return json.loads(text)
        except Exception:
            return {'raw': text}
    return {}

In [12]:
import asyncio
import json
import os
import subprocess
import sys

import ollama
from mcp.client.sse import sse_client
from mcp import ClientSession

MODEL = "qwen3:8b"
#MODEL = "llama3.2:latest"
#MODEL = "llama3:8b"


def _get(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


async def start_mcp_server(script_path='mcp/server.py'):
    if not os.path.exists(script_path):
        raise FileNotFoundError(f'MCP server script not found: {script_path}')
    proc = subprocess.Popen([sys.executable, script_path], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    await asyncio.sleep(2)
    return proc


async def connect_to_mcp(server_url='http://127.0.0.1:8000/sse'):
    ctx = sse_client(server_url)
    read_stream, write_stream = await ctx.__aenter__()
    session = ClientSession(read_stream, write_stream)
    await session.__aenter__()
    await asyncio.wait_for(session.initialize(), timeout=10)
    return session, ctx


def mcp_tools_to_ollama(mcp_tools):
    converted = []
    for tool in mcp_tools:
        converted.append({
            'type': 'function',
            'function': {
                'name': tool.name,
                'description': tool.description,
                'parameters': getattr(tool, 'inputSchema', None) or {'type': 'object', 'properties': {}},
            },
        })
    return converted


def coerce_tool_arguments(tool_args, proposals):
    if not isinstance(tool_args, dict):
        tool_args = {}
    if not tool_args.get('proposals'):
        tool_args = dict(tool_args)
        tool_args['proposals'] = proposals
    return tool_args


async def main():
    proc = await start_mcp_server('mcp/server.py')
    print('MCP server started (pid=', proc.pid, ')')
    try:
        session, ctx = await connect_to_mcp()
        print('Connected to MCP server')
        tools_result = await session.list_tools()
        ollama_tools = mcp_tools_to_ollama(tools_result.tools)
        print('Available tools:', [tool.name for tool in tools_result.tools])

        current_proposals = {'1': {'from': 'Gent', 'to': 'Antwerp', 'time_leave': '08:00'}}
        system_content = (
            'You are an EV mobility planning assistant. Use the available tools when a trip distance or arrival time is needed. '
            'When calling a tool, include a proposals object shaped like {"1": {"from": ..., "to": ..., "time_leave": ...}}. '
            'Do not ask the user for trip-derived values if a tool can compute them.'
        )
        messages = [
            {'role': 'system', 'content': system_content},
            {'role': 'user', 'content': 'Plan this trip: Gent to Antwerp, departure 08:00. Return JSON tool_calls using proposals.'},
        ]

        iteration = 0
        while True:
            iteration += 1
            print(f'\n--- Loop iteration {iteration} ---')
            response = ollama.chat(model=MODEL, messages=messages, tools=ollama_tools)
            message = _get(response, 'message', {})
            tool_calls = _get(message, 'tool_calls', []) or []
            print('Assistant text:', _get(message, 'content', ''))
            print('Tool calls:', tool_calls)
            messages.append(message)

            if not tool_calls:
                print('No tool calls returned; stopping.')
                break

            for tool_call in tool_calls:
                function = _get(tool_call, 'function', {})
                tool_name = _get(function, 'name', 'unknown')
                tool_args = coerce_tool_arguments(_get(function, 'arguments', {}), current_proposals)
                print(f'Calling MCP tool: {tool_name}')
                print('Arguments:', json.dumps(tool_args, indent=2))
                tool_result = await session.call_tool(tool_name, arguments=tool_args)
                result_text = tool_result.content[0].text if tool_result.content else '{}'
                print('Result:', result_text)
                try:
                    parsed_result = json.loads(result_text)
                    if isinstance(parsed_result, dict) and parsed_result.get('proposals'):
                        current_proposals = parsed_result['proposals']
                except Exception:
                    pass
                messages.append({'role': 'tool', 'name': tool_name, 'content': result_text})

        print('\nFinal assistant message:')
        print(_get(messages[-1], 'content', ''))
        await session.__aexit__(None, None, None)
        await ctx.__aexit__(None, None, None)
    finally:
        try:
            proc.terminate()
            proc.wait(timeout=5)
        except Exception:
            try:
                proc.kill()
            except Exception:
                pass
        print('MCP server stopped')


# Run the smoke test
await main()

MCP server started (pid= 16228 )
Connected to MCP server
Available tools: ['fill_trip_arrival_times', 'fill_trip_distances']

--- Loop iteration 1 ---
Assistant text: 
Tool calls: [ToolCall(function=Function(name='fill_trip_arrival_times', arguments={'proposals': {'1': {'from': 'Gent', 'time_leave': '08:00', 'to': 'Antwerp'}}}))]
Calling MCP tool: fill_trip_arrival_times
Arguments: {
  "proposals": {
    "1": {
      "from": "Gent",
      "time_leave": "08:00",
      "to": "Antwerp"
    }
  }
}
Result: {
  "error": "openrouteservice helper unavailable: No module named 'openrouteservice_mcp'",
  "proposals": {
    "1": {
      "from": "Gent",
      "time_leave": "08:00",
      "to": "Antwerp"
    }
  }
}

--- Loop iteration 2 ---
Assistant text: The OpenRouteService helper is unavailable due to a missing module. Would you like me to:
1. Attempt the distance calculation instead using `fill_trip_distances`?
2. Suggest alternative routing methods?
3. Help with something else related to you